# SQL Outputs to Python

This notebook connects to the local PostgreSQL analytical database, exports final SQL result sets to CSV, and generates chart assets for the project README.

## 1. Setup and database connection

In [ ]:
# Core libraries for database extraction, file handling, and chart generation
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, text
import matplotlib.pyplot as plt


In [ ]:
import os

DB_USER = os.getenv("POSTGRES_USER", "postgres")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD")
DB_HOST = os.getenv("POSTGRES_HOST", "localhost")
DB_PORT = os.getenv("POSTGRES_PORT", "5433")
DB_NAME = os.getenv("POSTGRES_DB", "postgres")

if DB_PASSWORD is None:
    raise ValueError("POSTGRES_PASSWORD environment variable is not set.")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

In [ ]:
# Resolve project folders and create output directories if they do not exist
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

EXPORT_DIR = PROJECT_ROOT / "outputs" / "python_exports"
CHART_DIR = PROJECT_ROOT / "readme_assets" / "charts"

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Export directory:", EXPORT_DIR.resolve())
print("Chart directory:", CHART_DIR.resolve())

## 2. Helper functions

In [ ]:
def read_query(query: str) -> pd.DataFrame:
    """
    Run a SQL query against PostgreSQL and return the result as a pandas DataFrame.
    """
    with engine.connect() as conn:
        return pd.read_sql_query(text(query), conn)


def export_query(query_name: str, query: str) -> pd.DataFrame:
    """
    Run a SQL query, export the result to CSV, and return the DataFrame.
    """
    df = read_query(query)
    output_path = EXPORT_DIR / f"{query_name}.csv"
    df.to_csv(output_path, index=False)
    print(f"Exported {query_name}: {len(df):,} rows -> {output_path}")
    return df

In [ ]:
test_query = """
SELECT COUNT(*) AS order_rows
FROM fact_orders_clean;
"""

test_df = read_query(test_query)
test_df

## 3. Export final SQL outputs

In [ ]:
# Store final portfolio-ready SQL outputs in a query registry.
# Each query produces one clean CSV export for charts or README assets.
queries = {}

In [ ]:
# Q-06: Segment-level GMV concentration
queries["q06_gmv_by_segment"] = """
WITH segment_gmv AS (
    SELECT
        cm.business_segment,
        ROUND(SUM(foi.item_gmv)::numeric, 2) AS segment_gmv,
        COUNT(*) AS order_items,
        COUNT(DISTINCT foi.order_id) AS orders_containing_segment
    FROM fact_order_items_clean AS foi
    LEFT JOIN category_mapping AS cm
        ON foi.product_category_english = cm.product_category_english
    WHERE foi.order_status NOT IN ('canceled', 'unavailable')
      AND foi.item_gmv IS NOT NULL
    GROUP BY
        cm.business_segment
),
segment_gmv_with_share AS (
    SELECT
        business_segment,
        segment_gmv,
        order_items,
        orders_containing_segment,
        ROUND(
            segment_gmv / SUM(segment_gmv) OVER (),
            4
        ) AS segment_gmv_share
    FROM segment_gmv
),
ranked_segments AS (
    SELECT
        business_segment,
        segment_gmv,
        order_items,
        orders_containing_segment,
        segment_gmv_share,
        ROUND(
            SUM(segment_gmv_share) OVER (
                ORDER BY segment_gmv DESC
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ),
            4
        ) AS cumulative_gmv_share,
        RANK() OVER (
            ORDER BY segment_gmv DESC
        ) AS gmv_rank
    FROM segment_gmv_with_share
)
SELECT
    business_segment,
    segment_gmv,
    segment_gmv_share,
    cumulative_gmv_share,
    order_items,
    orders_containing_segment,
    gmv_rank
FROM ranked_segments
ORDER BY
    segment_gmv DESC;
"""

In [ ]:
q06_gmv_by_segment = export_query(
    "q06_gmv_by_segment",
    queries["q06_gmv_by_segment"]
)

q06_gmv_by_segment

In [ ]:
# Q-12: Seller-side regional performance
queries["q12_regional_seller_performance"] = """
WITH seller_order_base AS (
    SELECT
        foi.seller_id,
        foi.order_id,
        MAX(foi.seller_state) AS seller_state,
        MAX(foi.order_status) AS order_status,
        MAX(foi.delivery_days) AS delivery_days,
        MAX(foi.late_delivery_flag) AS late_delivery_flag,
        SUM(foi.item_gmv) AS seller_order_gmv
    FROM fact_order_items_clean AS foi
    GROUP BY
        foi.seller_id,
        foi.order_id
),
seller_order_review AS (
    SELECT
        sob.seller_id,
        sob.order_id,
        sob.seller_state,
        sob.delivery_days,
        sob.late_delivery_flag,
        sob.seller_order_gmv,
        rs.avg_review_score
    FROM seller_order_base AS sob
    LEFT JOIN review_summary AS rs
        ON sob.order_id = rs.order_id
    WHERE sob.order_status = 'delivered'
),
region_metrics AS (
    SELECT
        bsr.region,
        COUNT(*) AS delivered_orders,
        COUNT(*) FILTER (WHERE sor.avg_review_score IS NOT NULL) AS reviewed_orders,
        COUNT(DISTINCT sor.seller_id) AS active_sellers,
        ROUND(AVG(sor.avg_review_score)::numeric, 2) AS avg_review_score,
        ROUND(
            100.0 * AVG(CASE WHEN sor.late_delivery_flag = 1 THEN 1.0 ELSE 0.0 END)::numeric,
            2
        ) AS late_delivery_rate_pct,
        ROUND(AVG(sor.delivery_days)::numeric, 2) AS avg_delivery_days,
        ROUND(SUM(sor.seller_order_gmv)::numeric, 2) AS seller_gmv
    FROM seller_order_review AS sor
    LEFT JOIN brazil_state_reference AS bsr
        ON sor.seller_state = bsr.state_code
    GROUP BY
        bsr.region
),
filtered_regions AS (
    SELECT
        region,
        active_sellers,
        delivered_orders,
        reviewed_orders,
        avg_review_score,
        late_delivery_rate_pct,
        avg_delivery_days,
        seller_gmv
    FROM region_metrics
    WHERE delivered_orders >= 100
      AND active_sellers >= 10
)
SELECT
    region,
    active_sellers,
    delivered_orders,
    reviewed_orders,
    avg_review_score,
    late_delivery_rate_pct,
    avg_delivery_days,
    seller_gmv
FROM filtered_regions
ORDER BY
    avg_review_score ASC,
    delivered_orders DESC;
"""

In [ ]:
q12_regional_seller_performance = export_query(
    "q12_regional_seller_performance",
    queries["q12_regional_seller_performance"]
)

q12_regional_seller_performance

In [ ]:
list(EXPORT_DIR.glob("*.csv"))

In [ ]:
queries["q01_late_delivery_review_by_state"] = """
WITH state_metrics AS (
    SELECT
        fo.customer_state,
        COUNT(*) AS delivered_orders,
        SUM(fo.late_delivery_flag) AS late_delivered_orders,
        ROUND(SUM(fo.late_delivery_flag)::numeric / COUNT(*), 4) AS late_delivery_rate,
        ROUND(AVG(rs.avg_review_score), 2) AS avg_review_score
    FROM fact_orders_clean fo
    LEFT JOIN review_summary rs
        ON fo.order_id = rs.order_id
    WHERE fo.order_status = 'delivered'
    GROUP BY
        fo.customer_state
)
SELECT
    customer_state,
    delivered_orders,
    late_delivered_orders,
    late_delivery_rate,
    avg_review_score
FROM state_metrics
ORDER BY
    late_delivery_rate DESC,
    delivered_orders DESC; 
"""

In [ ]:
q01_late_delivery_review_by_state = export_query(
    "q01_late_delivery_review_by_state",
    queries["q01_late_delivery_review_by_state"]
)

q01_late_delivery_review_by_state

In [ ]:
queries["q01_late_delivery_review_by_region"] = """
WITH region_operational_metrics AS (
    SELECT
        bsr.region,
        COUNT(*) AS delivered_orders,
        SUM(foc.late_delivery_flag) AS late_delivered_orders,
        COUNT(*) FILTER (WHERE rs.avg_review_score IS NOT NULL) AS reviewed_orders,
        ROUND(
            100.0 * SUM(foc.late_delivery_flag)::numeric / COUNT(*),
            2
        ) AS late_delivery_rate_pct,
        ROUND(AVG(rs.avg_review_score)::numeric, 2) AS avg_review_score,
        ROUND(AVG(foc.delivery_days)::numeric, 2) AS avg_delivery_days
    FROM fact_orders_clean AS foc
    LEFT JOIN review_summary AS rs
        ON foc.order_id = rs.order_id
    LEFT JOIN brazil_state_reference AS bsr
        ON foc.customer_state = bsr.state_code
    WHERE foc.order_status = 'delivered'
    GROUP BY
        bsr.region
)
SELECT
    region,
    delivered_orders,
    late_delivered_orders,
    reviewed_orders,
    late_delivery_rate_pct,
    avg_review_score,
    avg_delivery_days
FROM region_operational_metrics
ORDER BY
    late_delivery_rate_pct DESC,
    delivered_orders DESC;
"""

In [ ]:
q01_late_delivery_review_by_region = export_query(
    "q01_late_delivery_review_by_region",
    queries["q01_late_delivery_review_by_region"]
)

q01_late_delivery_review_by_region

In [ ]:
queries["q08_gmv_basket_by_region"] = """
WITH region_basket_metrics AS (
    SELECT
        bsr.region,
        COUNT(*) AS total_orders,
        ROUND(SUM(foc.order_gmv)::numeric, 2) AS total_gmv,
        ROUND(AVG(foc.order_gmv)::numeric, 2) AS avg_order_value,
        ROUND(AVG(foc.total_items)::numeric, 2) AS avg_items_per_order
    FROM fact_orders_clean AS foc
    LEFT JOIN brazil_state_reference AS bsr
        ON foc.customer_state = bsr.state_code
    WHERE foc.order_status NOT IN ('canceled', 'unavailable')
      AND foc.order_gmv IS NOT NULL
    GROUP BY
        bsr.region
)
SELECT
    region,
    total_orders,
    total_gmv,
    avg_order_value,
    avg_items_per_order
FROM region_basket_metrics
ORDER BY
    total_gmv DESC;
"""

q08_gmv_basket_by_region = export_query(
    "q08_gmv_basket_by_region",
    queries["q08_gmv_basket_by_region"]
)

q08_gmv_basket_by_region

In [ ]:
queries["q10_seller_concentration_summary"] = """
WITH seller_gmv AS (
    SELECT
        seller_id,
        SUM(item_gmv) AS seller_gmv
    FROM fact_order_items_clean
    WHERE order_status NOT IN ('canceled', 'unavailable')
      AND item_gmv IS NOT NULL
    GROUP BY
        seller_id
),
ranked_sellers AS (
    SELECT
        seller_id,
        seller_gmv,
        ROW_NUMBER() OVER (ORDER BY seller_gmv DESC) AS seller_rank,
        COUNT(*) OVER () AS total_sellers,
        SUM(seller_gmv) OVER () AS total_marketplace_gmv
    FROM seller_gmv
),
seller_thresholds AS (
    SELECT
        *,
        CEIL(total_sellers * 0.10) AS top_10_cutoff,
        CEIL(total_sellers * 0.20) AS top_20_cutoff
    FROM ranked_sellers
)
SELECT
    MAX(total_sellers) AS total_sellers,
    COUNT(*) FILTER (WHERE seller_rank <= top_10_cutoff) AS top_10_seller_count,
    ROUND(SUM(seller_gmv) FILTER (WHERE seller_rank <= top_10_cutoff)::numeric, 2) AS top_10_gmv,
    ROUND(
        100.0 * SUM(seller_gmv) FILTER (WHERE seller_rank <= top_10_cutoff)
        / MAX(total_marketplace_gmv)::numeric,
        2
    ) AS top_10_gmv_share_pct,
    COUNT(*) FILTER (WHERE seller_rank <= top_20_cutoff) AS top_20_seller_count,
    ROUND(SUM(seller_gmv) FILTER (WHERE seller_rank <= top_20_cutoff)::numeric, 2) AS top_20_gmv,
    ROUND(
        100.0 * SUM(seller_gmv) FILTER (WHERE seller_rank <= top_20_cutoff)
        / MAX(total_marketplace_gmv)::numeric,
        2
    ) AS top_20_gmv_share_pct,
    ROUND(MAX(total_marketplace_gmv)::numeric, 2) AS total_marketplace_gmv
FROM seller_thresholds;
"""

q10_seller_concentration_summary = export_query(
    "q10_seller_concentration_summary",
    queries["q10_seller_concentration_summary"]
)

q10_seller_concentration_summary

In [ ]:
queries["q09_seller_quality_quadrant"] = """
WITH seller_order_base AS (
    SELECT
        foi.seller_id,
        foi.order_id,
        MAX(foi.seller_state) AS seller_state,
        MAX(foi.order_status) AS order_status,
        SUM(foi.item_gmv) AS seller_order_gmv
    FROM fact_order_items_clean AS foi
    GROUP BY
        foi.seller_id,
        foi.order_id
),
seller_review_base AS (
    SELECT
        sob.seller_id,
        MAX(sob.seller_state) AS seller_state,
        COUNT(*) AS delivered_orders,
        COUNT(*) FILTER (WHERE rs.avg_review_score IS NOT NULL) AS reviewed_orders,
        SUM(sob.seller_order_gmv) AS seller_gmv,
        AVG(rs.avg_review_score) AS avg_review_score
    FROM seller_order_base AS sob
    LEFT JOIN review_summary AS rs
        ON sob.order_id = rs.order_id
    WHERE sob.order_status = 'delivered'
    GROUP BY
        sob.seller_id
),
seller_filtered AS (
    SELECT
        *
    FROM seller_review_base
    WHERE reviewed_orders >= 20
),
seller_benchmarks AS (
    SELECT
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY delivered_orders) AS high_volume_threshold,
        AVG(avg_review_score) AS avg_review_score_threshold
    FROM seller_filtered
),
seller_quadrants AS (
    SELECT
        sf.seller_id,
        sf.seller_state,
        sf.delivered_orders,
        sf.reviewed_orders,
        sf.seller_gmv,
        sf.avg_review_score,
        CASE
            WHEN sf.delivered_orders >= sb.high_volume_threshold
             AND sf.avg_review_score >= sb.avg_review_score_threshold
                THEN 'High volume / High rating'
            WHEN sf.delivered_orders >= sb.high_volume_threshold
             AND sf.avg_review_score < sb.avg_review_score_threshold
                THEN 'High volume / Low rating'
            WHEN sf.delivered_orders < sb.high_volume_threshold
             AND sf.avg_review_score >= sb.avg_review_score_threshold
                THEN 'Low volume / High rating'
            ELSE 'Low volume / Low rating'
        END AS seller_quadrant
    FROM seller_filtered AS sf
    CROSS JOIN seller_benchmarks AS sb
),
quadrant_metrics AS (
    SELECT
        seller_quadrant,
        COUNT(*) AS seller_count,
        ROUND(SUM(seller_gmv)::numeric, 2) AS total_gmv,
        ROUND(
            100.0 * SUM(seller_gmv) / SUM(SUM(seller_gmv)) OVER (),
            2
        ) AS gmv_share_pct,
        ROUND(AVG(avg_review_score)::numeric, 2) AS avg_review_score,
        ROUND(AVG(delivered_orders)::numeric, 2) AS avg_delivered_orders
    FROM seller_quadrants
    GROUP BY
        seller_quadrant
)
SELECT
    seller_quadrant,
    seller_count,
    total_gmv,
    gmv_share_pct,
    avg_review_score,
    avg_delivered_orders
FROM quadrant_metrics
ORDER BY
    total_gmv DESC;
"""

q09_seller_quality_quadrant = export_query(
    "q09_seller_quality_quadrant",
    queries["q09_seller_quality_quadrant"]
)

q09_seller_quality_quadrant

In [ ]:
queries["q11_segment_review_delivery"] = """
WITH order_segment_base AS (
    SELECT
        cm.business_segment,
        foi.order_id,
        MAX(foi.order_status) AS order_status,
        MAX(foi.late_delivery_flag) AS late_delivery_flag,
        MAX(foi.delivery_days) AS delivery_days,
        SUM(foi.item_gmv) AS segment_order_gmv
    FROM fact_order_items_clean AS foi
    LEFT JOIN category_mapping AS cm
        ON foi.product_category_english = cm.product_category_english
    WHERE foi.order_status = 'delivered'
      AND foi.item_gmv IS NOT NULL
      AND cm.business_segment IS NOT NULL
    GROUP BY
        cm.business_segment,
        foi.order_id
)
SELECT
    osb.business_segment,
    COUNT(*) AS delivered_orders_containing_segment,
    COUNT(*) FILTER (WHERE rs.avg_review_score IS NOT NULL) AS reviewed_orders,
    ROUND(AVG(rs.avg_review_score)::numeric, 2) AS avg_review_score,
    ROUND(
        100.0 * AVG(
            CASE WHEN osb.late_delivery_flag = 1 THEN 1.0 ELSE 0.0 END
        )::numeric,
        2
    ) AS late_delivery_rate_pct,
    ROUND(AVG(osb.delivery_days)::numeric, 2) AS avg_delivery_days,
    ROUND(SUM(osb.segment_order_gmv)::numeric, 2) AS segment_gmv
FROM order_segment_base AS osb
LEFT JOIN review_summary AS rs
    ON osb.order_id = rs.order_id
GROUP BY
    osb.business_segment
ORDER BY
    avg_review_score ASC,
    segment_gmv DESC;
"""

q11_segment_review_delivery = export_query(
    "q11_segment_review_delivery",
    queries["q11_segment_review_delivery"]
)

q11_segment_review_delivery

In [ ]:
sorted([p.name for p in EXPORT_DIR.glob("*.csv")])

In [ ]:
expected_exports = {
    "q01_late_delivery_review_by_region.csv",
    "q01_late_delivery_review_by_state.csv",
    "q04_orders_by_hour.csv",
    "q06_gmv_by_segment.csv",
    "q08_gmv_basket_by_region.csv",
    "q09_seller_quality_quadrant.csv",
    "q10_seller_concentration_summary.csv",
    "q11_segment_review_delivery.csv",
    "q12_regional_seller_performance.csv",
}

actual_exports = {p.name for p in EXPORT_DIR.glob("*.csv")}

missing_exports = expected_exports - actual_exports
extra_exports = actual_exports - expected_exports

print("Export directory:", EXPORT_DIR.resolve())
print("Expected exports:", len(expected_exports))
print("Actual exports:", len(actual_exports))
print("Missing exports:", missing_exports)
print("Extra exports:", extra_exports)

## 4. Generate portfolio charts

In [ ]:
# Q-06 chart: show which business segments contribute the largest GMV share
q06 = pd.read_csv(EXPORT_DIR / "q06_gmv_by_segment.csv")

q06

In [ ]:
q06_plot = q06.sort_values("segment_gmv", ascending=True).copy()

q06_plot["gmv_share_pct"] = q06_plot["segment_gmv_share"] * 100

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.barh(
    q06_plot["business_segment"],
    q06_plot["segment_gmv"]
)

ax.set_title("GMV by Business Segment")
ax.set_xlabel("GMV")
ax.set_ylabel("Business Segment")

# Add GMV share labels at the end of each bar
for i, row in q06_plot.iterrows():
    ax.text(
        row["segment_gmv"],
        row["business_segment"],
        f" {row['gmv_share_pct']:.1f}%",
        va="center"
    )

ax.ticklabel_format(style="plain", axis="x")
plt.tight_layout()

output_path = CHART_DIR / "q06_gmv_by_segment.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved chart to: {output_path}")

In [ ]:
list(CHART_DIR.glob("*.png"))

In [ ]:
q01_region = pd.read_csv(EXPORT_DIR / "q01_late_delivery_review_by_region.csv")

q01_region

In [ ]:
q01_region_plot = q01_region.sort_values(
    "late_delivery_rate_pct",
    ascending=True
).copy()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.barh(
    q01_region_plot["region"],
    q01_region_plot["late_delivery_rate_pct"]
)

ax.set_title("Late Delivery Rate by Customer Region")
ax.set_xlabel("Late Delivery Rate (%)")
ax.set_ylabel("Customer Region")

for _, row in q01_region_plot.iterrows():
    ax.text(
        row["late_delivery_rate_pct"],
        row["region"],
        f" {row['late_delivery_rate_pct']:.1f}%",
        va="center"
    )

plt.tight_layout()

output_path = CHART_DIR / "q01_late_delivery_by_region.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved chart to: {output_path}")

In [ ]:
sorted([p.name for p in CHART_DIR.glob("*.png")])

In [ ]:
q08_region = pd.read_csv(EXPORT_DIR / "q08_gmv_basket_by_region.csv")

q08_region

In [ ]:
q08_plot = q08_region.copy()

q08_plot["total_gmv_millions"] = q08_plot["total_gmv"] / 1_000_000

# Scale bubble sizes so large regions stand out but do not overwhelm the chart
q08_plot["bubble_size"] = (
    q08_plot["total_orders"] / q08_plot["total_orders"].max()
) * 2500

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(
    q08_plot["total_gmv_millions"],
    q08_plot["avg_order_value"],
    s=q08_plot["bubble_size"],
    alpha=0.7
)

for _, row in q08_plot.iterrows():
    ax.text(
        row["total_gmv_millions"],
        row["avg_order_value"],
        f" {row['region']}",
        va="center"
    )

ax.set_title("Regional GMV vs Average Order Value")
ax.set_xlabel("Total GMV (BRL millions)")
ax.set_ylabel("Average Order Value (BRL)")

plt.tight_layout()

output_path = CHART_DIR / "q08_region_gmv_vs_aov.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved chart to: {output_path}")

In [ ]:
sorted([p.name for p in CHART_DIR.glob("*.png")])

In [ ]:
q09 = pd.read_csv(EXPORT_DIR / "q09_seller_quality_quadrant.csv")

q09

In [ ]:
q09_plot = q09.sort_values("gmv_share_pct", ascending=True).copy()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.barh(
    q09_plot["seller_quadrant"],
    q09_plot["gmv_share_pct"]
)

ax.set_title("GMV Share by Seller Quality Quadrant")
ax.set_xlabel("GMV Share (%)")
ax.set_ylabel("Seller Quality Quadrant")

for _, row in q09_plot.iterrows():
    ax.text(
        row["gmv_share_pct"],
        row["seller_quadrant"],
        f" {row['gmv_share_pct']:.1f}% | {int(row['seller_count'])} sellers",
        va="center"
    )

plt.tight_layout()

output_path = CHART_DIR / "q09_seller_quality_quadrant.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved chart to: {output_path}")

In [ ]:
sorted([p.name for p in CHART_DIR.glob("*.png")])

In [ ]:
q10 = pd.read_csv(EXPORT_DIR / "q10_seller_concentration_summary.csv")

q10

In [ ]:
q10_plot = pd.DataFrame({
    "seller_group": ["Top 10% Sellers", "Top 20% Sellers"],
    "gmv_share_pct": [
        q10.loc[0, "top_10_gmv_share_pct"],
        q10.loc[0, "top_20_gmv_share_pct"]
    ],
    "seller_count": [
        q10.loc[0, "top_10_seller_count"],
        q10.loc[0, "top_20_seller_count"]
    ]
})

q10_plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(
    q10_plot["seller_group"],
    q10_plot["gmv_share_pct"]
)

ax.set_title("GMV Concentration by Top Seller Cohorts")
ax.set_ylabel("GMV Share (%)")
ax.set_xlabel("Seller Cohort")
ax.set_ylim(0, 100)

for _, row in q10_plot.iterrows():
    ax.text(
        row["seller_group"],
        row["gmv_share_pct"],
        f"{row['gmv_share_pct']:.1f}%\n{int(row['seller_count'])} sellers",
        ha="center",
        va="bottom"
    )

plt.tight_layout()

output_path = CHART_DIR / "q10_seller_concentration.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved chart to: {output_path}")

In [ ]:
sorted([p.name for p in CHART_DIR.glob("*.png")])

In [ ]:
q12 = pd.read_csv(EXPORT_DIR / "q12_regional_seller_performance.csv")

q12

In [ ]:
q12_plot = q12.copy()

q12_plot["seller_gmv_millions"] = q12_plot["seller_gmv"] / 1_000_000

q12_plot["bubble_size"] = (
    q12_plot["seller_gmv"] / q12_plot["seller_gmv"].max()
) * 3000

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(
    q12_plot["late_delivery_rate_pct"],
    q12_plot["avg_review_score"],
    s=q12_plot["bubble_size"],
    alpha=0.7
)

for _, row in q12_plot.iterrows():
    ax.text(
        row["late_delivery_rate_pct"],
        row["avg_review_score"],
        f" {row['region']}",
        va="center"
    )

ax.set_title("Regional Seller Performance: Review Score vs Late Delivery")
ax.set_xlabel("Late Delivery Rate (%)")
ax.set_ylabel("Average Review Score")

plt.tight_layout()

output_path = CHART_DIR / "q12_regional_seller_performance.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved chart to: {output_path}")

In [ ]:
sorted([p.name for p in CHART_DIR.glob("*.png")])

In [ ]:
queries["q04_orders_by_hour"] = """
SELECT
    EXTRACT(HOUR FROM order_purchase_timestamp)::int AS order_hour,
    COUNT(*) AS order_count
FROM fact_orders_clean
GROUP BY
    EXTRACT(HOUR FROM order_purchase_timestamp)::int
ORDER BY
    order_hour;
"""

q04_orders_by_hour = export_query(
    "q04_orders_by_hour",
    queries["q04_orders_by_hour"]
)

q04_orders_by_hour

In [ ]:
q04_plot = pd.read_csv(EXPORT_DIR / "q04_orders_by_hour.csv")
q04_plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    q04_plot["order_hour"],
    q04_plot["order_count"],
    marker="o"
)

ax.set_title("Order Volume by Hour of Day")
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Order Count")
ax.set_xticks(range(0, 24))

plt.tight_layout()

output_path = CHART_DIR / "q04_orders_by_hour.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved chart to: {output_path}")

In [ ]:
sorted([p.name for p in CHART_DIR.glob("*.png")])

## 5. Validate exported files

In [ ]:
expected_exports = {
    "q01_late_delivery_review_by_region.csv",
    "q01_late_delivery_review_by_state.csv",
    "q04_orders_by_hour.csv",
    "q06_gmv_by_segment.csv",
    "q08_gmv_basket_by_region.csv",
    "q09_seller_quality_quadrant.csv",
    "q10_seller_concentration_summary.csv",
    "q12_regional_seller_performance.csv",
}

expected_charts = {
    "q01_late_delivery_by_region.png",
    "q04_orders_by_hour.png",
    "q06_gmv_by_segment.png",
    "q08_region_gmv_vs_aov.png",
    "q09_seller_quality_quadrant.png",
    "q10_seller_concentration.png",
    "q12_regional_seller_performance.png",
}

actual_exports = {p.name for p in EXPORT_DIR.glob("*.csv")}
actual_charts = {p.name for p in CHART_DIR.glob("*.png")}

missing_exports = expected_exports - actual_exports
missing_charts = expected_charts - actual_charts

extra_exports = actual_exports - expected_exports
extra_charts = actual_charts - expected_charts

print("Export directory:", EXPORT_DIR.resolve())
print("Chart directory:", CHART_DIR.resolve())

print("\nExpected exports:", len(expected_exports))
print("Actual exports:", len(actual_exports))
print("Missing exports:", missing_exports)
print("Extra exports:", extra_exports)

print("\nExpected charts:", len(expected_charts))
print("Actual charts:", len(actual_charts))
print("Missing charts:", missing_charts)
print("Extra charts:", extra_charts)

In [ ]:
chart_files = sorted([p.name for p in CHART_DIR.glob("*.png")])
export_files = sorted([p.name for p in EXPORT_DIR.glob("*.csv")])

print("Charts:")
for file in chart_files:
    print("-", file)

print("\nCSV exports:")
for file in export_files:
    print("-", file)